# Fine-tune GPT-2 Chat Model

Fine-tunes GPT-2 (124M params) on real chat data. Produces a model that can hold conversations.

**Instructions:**
1. Runtime > Change runtime type > **T4 GPU**
2. Runtime > Run all
3. Takes ~20-30 minutes total

In [ ]:
!pip install datasets tiktoken peft -q
print('Done!')

In [ ]:
import torch, time, math, json
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2LMHeadModel, GPT2TokenizerFast, get_linear_schedule_with_warmup
from datasets import load_dataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# Load GPT-2 and tokenizer
MODEL_NAME = 'gpt2'
tokenizer = GPT2TokenizerFast.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)
model.config.pad_token_id = tokenizer.pad_token_id

n_params = sum(p.numel() for p in model.parameters())
print(f'Loaded {MODEL_NAME}: {n_params/1e6:.0f}M parameters')

In [ ]:
# Load Guanaco chat dataset
print('Loading chat dataset...')
ds = load_dataset('mlabonne/guanaco-llama2-1k', split='train')

CONV_TEMPLATE = """<|user|>
{user}
<|assistant|>
{assistant}<|end|>"""

def format_conversation(text):
    parts = text.split('### ')
    turns = []
    for part in parts:
        part = part.strip()
        if part.startswith('Human:'):
            turns.append(('user', part[len('Human:'):].strip()))
        elif part.startswith('Assistant:'):
            turns.append(('assistant', part[len('Assistant:'):].strip()))
    if len(turns) < 2:
        return None
    formatted = ''
    for role, content in turns:
        if role == 'user':
            formatted += f'<|user|>\n{content}\n<|assistant|>\n'
        else:
            formatted += f'{content}<|end|>\n'
    return formatted

texts = []
for example in ds:
    result = format_conversation(example['text'])
    if result and len(result) > 20:
        texts.append(result)

print(f'Formatted {len(texts)} conversations')

In [ ]:
class ChatFTDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=512):
        self.examples = []
        for text in texts:
            tokens = tokenizer.encode(text, truncation=True, max_length=max_len)
            if len(tokens) > 10:
                self.examples.append(tokens)

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        tokens = self.examples[idx]
        x = torch.tensor(tokens[:-1], dtype=torch.long)
        y = torch.tensor(tokens[1:], dtype=torch.long)
        return {'input_ids': x, 'labels': y}

def collate_fn(batch):
    max_len = max(b['input_ids'].size(0) for b in batch)
    pad_id = tokenizer.pad_token_id
    input_ids = torch.full((len(batch), max_len), pad_id, dtype=torch.long)
    labels = torch.full((len(batch), max_len), -100, dtype=torch.long)
    for i, b in enumerate(batch):
        L = b['input_ids'].size(0)
        input_ids[i, :L] = b['input_ids']
        labels[i, :L] = b['labels']
    return {'input_ids': input_ids, 'labels': labels}

BATCH_SIZE = 4
SEQ_LEN = 512

dataset = ChatFTDataset(texts, tokenizer, max_len=SEQ_LEN)
val_size = max(1, len(dataset) // 10)
train_size = len(dataset) - val_size

from torch.utils.data import random_split
train_ds, val_ds = random_split(dataset, [train_size, val_size])
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=0)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

In [ ]:
# Fine-tune
NUM_EPOCHS = 2
LR = 2e-5
GRAD_ACCUM = 4

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = (len(train_loader) // GRAD_ACCUM) * NUM_EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=total_steps // 10, num_training_steps=total_steps)

model.to(device)
scaler = torch.amp.GradScaler('cuda')

print(f'Training: {total_steps} steps, {NUM_EPOCHS} epochs')
print(f'Estimated time: ~{total_steps * 0.5 / 60:.0f} minutes\n')

global_step = 0
best_val_loss = float('inf')

for epoch in range(1, NUM_EPOCHS + 1):
    print(f'--- Epoch {epoch}/{NUM_EPOCHS} ---')
    model.train()
    total_loss = 0
    t0 = time.time()

    for step, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)

        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            out = model(input_ids=input_ids, labels=labels)
            loss = out.loss / GRAD_ACCUM

        scaler.scale(loss).backward()
        total_loss += loss.item() * GRAD_ACCUM

        if (step + 1) % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_step += 1

        if (step + 1) % (GRAD_ACCUM * 20) == 0:
            avg = total_loss / (step + 1)
            elapsed = time.time() - t0
            steps_per_sec = (step + 1) / elapsed
            eta = (len(train_loader) - step - 1) / steps_per_sec
            lr_now = scheduler.get_last_lr()[0]
            print(f'  Step {step+1}/{len(train_loader)} | Loss: {avg:.4f} | LR: {lr_now:.2e} | {steps_per_sec:.1f} steps/s | ETA: {eta/60:.1f}min')

    # Validate
    model.eval()
    val_loss = 0
    n = 0
    with torch.no_grad():
        for batch in val_loader:
            out = model(input_ids=batch['input_ids'].to(device), labels=batch['labels'].to(device))
            val_loss += out.loss.item()
            n += 1
    val_loss /= max(n, 1)
    print(f'  Val Loss: {val_loss:.4f} | PPL: {math.exp(val_loss):.2f}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        model.save_pretrained('chat_gpt2')
        tokenizer.save_pretrained('chat_gpt2')
        print('  Saved best model!')

print(f'\nDone! Best val loss: {best_val_loss:.4f} | PPL: {math.exp(best_val_loss):.2f}')

In [ ]:
# Test generation
model.eval()

def chat_generate(user_input, max_new_tokens=200, temperature=0.7, top_p=0.9):
    prompt = f'<|user|>\n{user_input}\n<|assistant|>\n'
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)

    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    response = tokenizer.decode(output[0], skip_special_tokens=False)
    reply = response.split('<|assistant|>\n')[-1].split('<|end|>')[0].strip()
    return reply

# Quick test
tests = ['What is Python?', 'Tell me a joke.', 'How do I learn coding?']
for q in tests:
    print(f'Q: {q}')
    print(f'A: {chat_generate(q)}\n')

In [ ]:
# Interactive chat
print('Chat with your fine-tuned GPT-2! Type quit to exit.\n')

while True:
    user_input = input('You: ').strip()
    if user_input.lower() in ('quit', 'exit', 'q'):
        break
    if not user_input:
        continue
    print(f'Assistant: {chat_generate(user_input)}\n')

In [ ]:
# Download model
import shutil, os
from google.colab import files

shutil.make_archive('chat_gpt2', 'zip', 'chat_gpt2')
files.download('chat_gpt2.zip')
print('Downloaded! Unzip and use with the local chat script.')